# 💉 01. PhoBERT Multitask Training - VaccineNLP Phase 5

Notebook này được thiết kế để huấn luyện mô hình PhoBERT đa nhiệm (Multitask) trên tập dữ liệu VaccineNLP tiếng Việt. 

**Mục tiêu:** Dự đoán đồng thời:
1. **Misinformation**: Tin giả / Không tin giả.
2. **Stance**: Quan điểm (Ủng hộ, Phản đối, Trung lập).
3. **Sentiment**: Sắc thái (Tích cực, Tiêu cực, Trung lập).

**Môi trường khuyến nghị:** GPU (Kaggle/Google Colab) hoặc Local có hỗ trợ CUDA.

In [1]:
# [CELL 1] Setup Môi trường
%pip install -q transformers[torch] datasets evaluate accelerate mlflow pyvi scikit-learn matplotlib seaborn
print("✅ Cài đặt môi trường hoàn tất!")

^C
Note: you may need to restart the kernel to use updated packages.
✅ Cài đặt môi trường hoàn tất!


In [ ]:
# [CELL 2] Imports & Cấu hình
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoConfig, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from pyvi import ViTokenizer
import mlflow
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đường dẫn
DATA_PATH = "../datasets/04_silver_labels/annotated_v6.jsonl"
BENCHMARK_SAVE_PATH = "../datasets/03_processed/benchmark_test_set.jsonl"
MODEL_SAVE_DIR = "../experiments/models/phobert-multitask-v1/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f"🚀 Đang sử dụng thiết bị: {DEVICE}")

In [ ]:
# [CELL 3] Load & Clean Data
def load_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if item.get('status') == 'llm_annotated':
                data.append({
                    "text": item['text_cleaned'],
                    "misinfo": item['llm_parsed_labels']['Misinformation'],
                    "stance": item['llm_parsed_labels']['Stance'],
                    "sentiment": item['llm_parsed_labels']['Sentiment']
                })
    return pd.DataFrame(data)

df = load_data(DATA_PATH)
print(f"📦 Đã load {len(df)} records.")
print(df.head(2))

In [ ]:
# [CELL 4] Word Segmentation (pyvi) & Tokenization
LABEL_MAPS = {
    "misinfo": {"Không chắc chắn": 0, "Không liên quan": 0, "Tin giả": 1, "Sai lệch": 1, "Không tin giả": 2, "Chính xác": 2},
    "stance": {"Ủng hộ": 0, "Phản đối": 1, "Trung lập": 2, "Không rõ": 3},
    "sentiment": {"Tiêu cực": 0, "Trung tính": 1, "Trung lập": 1, "Tích cực": 2}
}

TOKENIZER_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def preprocess_text(text):
    # PhoBERT yêu cầu văn bản đã được tách từ (Word Segmented)
    return ViTokenizer.tokenize(text)

df['text_segmented'] = df['text'].apply(preprocess_text)
df['misinfo_id'] = df['misinfo'].map(LABEL_MAPS['misinfo'])
df['stance_id'] = df['stance'].map(LABEL_MAPS['stance'])
df['sentiment_id'] = df['sentiment'].map(LABEL_MAPS['sentiment'])

print("✅ Tiền xử lý hoàn tất!")

In [ ]:
# [CELL 5] Splitting & BENCHMARK EXPORT (P0)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['stance_id'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['stance_id'])

print(f"📊 Số lượng: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Xuất Benchmark Test Set (JSONL định dạng chuẩn)
test_df.to_json(BENCHMARK_SAVE_PATH, orient='records', lines=True, force_ascii=False)
print(f"💾 Đã lưu Benchmark Test Set tại: {BENCHMARK_SAVE_PATH}")

In [ ]:
# [CELL 6] Model Architecture (Multitask)
class VaccineMultitaskModel(nn.Module):
    def __init__(self, model_name=TOKENIZER_NAME, n_misinfo=3, n_stance=4, n_sentiment=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        
        self.head_misinfo = nn.Linear(hidden, n_misinfo)
        self.head_stance = nn.Linear(hidden, n_stance)
        self.head_sentiment = nn.Linear(hidden, n_sentiment)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Pooler output hoặc CLS token (PhoBERT-base v2 pooler ổn)
        pooled = self.dropout(out.pooler_output)
        
        return {
            "misinfo": self.head_misinfo(pooled),
            "stance": self.head_stance(pooled),
            "sentiment": self.head_sentiment(pooled)
        }

# Loss function kết hợp class weights (Hard constraint: balanced)
from sklearn.utils.class_weight import compute_class_weight

def get_loss_fn(df, task_key, n_classes, device):
    weights = compute_class_weight('balanced', classes=np.arange(n_classes), y=df[task_key])
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))

loss_m = get_loss_fn(train_df, 'misinfo_id', 3, DEVICE)
loss_st = get_loss_fn(train_df, 'stance_id', 4, DEVICE)
loss_se = get_loss_fn(train_df, 'sentiment_id', 3, DEVICE)

print("✅ Khởi tạo kiến trúc và Loss hoàn tất!")

In [ ]:
# [CELL 7] Dataset & Training Loop
class VaccineDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=256):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        encoding = self.tokenizer(
            row['text_segmented'],
            truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'misinfo': torch.tensor(row['misinfo_id'], dtype=torch.long),
            'stance': torch.tensor(row['stance_id'], dtype=torch.long),
            'sentiment': torch.tensor(row['sentiment_id'], dtype=torch.long)
        }

# DataLoaders
train_loader = DataLoader(VaccineDataset(train_df, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(VaccineDataset(val_df, tokenizer), batch_size=16)

# Optimizer
model = VaccineMultitaskModel().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader, optimizer):
    model.train()
    losses = []
    for batch in tqdm(loader):
        optimizer.zero_grad()
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        
        logits = model(ids, mask)
        
        l1 = loss_m(logits['misinfo'], batch['misinfo'].to(DEVICE))
        l2 = loss_st(logits['stance'], batch['stance'].to(DEVICE))
        l3 = loss_se(logits['sentiment'], batch['sentiment'].to(DEVICE))
        
        total_loss = 0.5*l1 + 0.3*l2 + 0.2*l3
        total_loss.backward()
        optimizer.step()
        losses.append(total_loss.item())
    return np.mean(losses)

# Chạy training (Ví dụ 1 epoch)
print("🏃 Bắt đầu huấn luyện...")
avg_loss = train_epoch(model, train_loader, optimizer)
print(f"🔥 Epoch 1 Loss: {avg_loss:.4f}")

In [ ]:
# [CELL 8] Evaluation & Save

def evaluate(model, loader):
    model.eval()
    all_preds = {'m': [], 'st': [], 'se': []}
    all_labels = {'m': [], 'st': [], 'se': []}
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            
            all_preds['m'].extend(logits['misinfo'].argmax(dim=1).cpu().tolist())
            all_preds['st'].extend(logits['stance'].argmax(dim=1).cpu().tolist())
            all_preds['se'].extend(logits['sentiment'].argmax(dim=1).cpu().tolist())
            
            all_labels['m'].extend(batch['misinfo'].tolist())
            all_labels['st'].extend(batch['stance'].tolist())
            all_labels['se'].extend(batch['sentiment'].tolist())
            
    print("--- MISINFORMATION ---")
    print(classification_report(all_labels['m'], all_preds['m']))
    print("--- STANCE ---")
    print(classification_report(all_labels['st'], all_preds['st']))
    print("--- SENTIMENT ---")
    print(classification_report(all_labels['se'], all_preds['se']))

evaluate(model, val_loader)

# Save Model
torch.save(model.state_dict(), os.path.join(MODEL_SAVE_DIR, "pytorch_model.bin"))
print(f"💾 Mô hình đã được lưu tại: {MODEL_SAVE_DIR}")

### 🏁 Kết thúc
Anh Hưng có thể copy file này lên Kaggle hoặc Colab để chạy huấn luyện GPU.